# Train the gas-cylinder detector on Google Colab

This notebook trains YOLOv8n on the CylinDeRS dataset with a free Colab GPU,
which is much faster than training on a laptop CPU.

Before you start:
1. Upload `cylinders_dataset.zip` to the root of your Google Drive (**My Drive**).
2. Check that **Runtime → Change runtime type** is set to **T4 GPU**.

Then choose **Runtime → Run all** and keep this tab open.

## 1. Check the GPU

This should show a **Tesla T4**. If it says `command not found`, choose
**Runtime → Change runtime type → T4 GPU** and run the notebook again.

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

The same version as `requirements.txt` in the repository.

In [ ]:
!pip install -q "ultralytics~=8.4.171"

## 3. Connect Google Drive

Colab asks for permission. The dataset is read from Drive and the results are
saved back to Drive, so they are kept even if Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Unzip the dataset

The images are unzipped to the Colab disk, which is much faster to read than Drive.
You should see `data.yaml`, `test`, `train` and `valid`.

In [ ]:
!unzip -q -o /content/drive/MyDrive/cylinders_dataset.zip -d /content/data
!ls /content/data

## 5. Train

The same model and image size as `src/model.py`, but with all the training images.
After every epoch the weights are saved to `My Drive/gas_cylinders_runs/colab/weights/`.
To get a better model, increase `time`.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/data/data.yaml",
    project="/content/drive/MyDrive/gas_cylinders_runs",  # Save to Google Drive.
    name="colab",
    exist_ok=True,
    time=0.75,  # Train for 0.75 hours (45 minutes); this replaces epochs.
    imgsz=480,  # Same image size as src/model.py.
    cache=True,  # Keep the images in RAM, so every epoch is faster.
)

## 6. Look at the results

The losses should go down, while precision, recall and mAP should go up.

In [ ]:
from IPython.display import Image

Image("/content/drive/MyDrive/gas_cylinders_runs/colab/results.png")

## 7. Use the model on your laptop

1. In Google Drive, open `gas_cylinders_runs/colab/weights/` and download `best.pt`.
2. Put it in the repository as `runs/colab/weights/best.pt`. The `runs/` folder is ignored by Git.
3. In `src/evaluate.py` and `src/track.py`, change `"train"` to `"colab"` in `WEIGHTS` and run them.

**If Colab disconnects during training**, run cells 1-4 again, then continue from the last epoch:

```python
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/gas_cylinders_runs/colab/weights/last.pt")
model.train(resume=True)
```